# XGBoost

In [11]:
import os
import json
import pandas as pd
import numpy as np
import scipy.sparse
import json


from xgboost import XGBClassifier, XGBRanker
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from itertools import product

In [12]:
def precision_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rec_k)

def recall_at_k(rec_k, rel_set):
    if len(rel_set) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rel_set)

def ndcg_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    dcg = 0.0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel_set), len(rec_k))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal + 1))
    return (dcg / idcg) if idcg > 0 else 0.0

def hit_score_at_k(rec_k, rel_set):
    cant_relevantes = set(rec_k).intersection(set(rel_set))
    return len(cant_relevantes)
    #return 1.0 if any((i in rel_set) for i in rec_k) else 0.0

def map_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    ap_sum = 0.0
    hits = 0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            hits += 1
            ap_sum += hits / rank
    return ap_sum / len(rel_set) if len(rel_set) > 0 else 0.0

def diversity_at_k(rec_k, info_videojuegos):
    generos_total = set()
    
    for app_id in rec_k:
        for genero in info_videojuegos[app_id]:
            generos_total.add(genero)

    if not generos_total:
        return 0
    
    return len(generos_total) 



def f1_at_k(rec_k, rel_set):
    if len(rec_k) == 0 or len(rel_set) == 0:
        return 0.0

    p = precision_at_k(rec_k, rel_set)
    r = recall_at_k(rec_k, rel_set)

    if (p + r) <= 0:
        return 0

    return 2 * p * r / (p + r)


In [13]:
base_dir = os.getcwd()
data_dir = os.path.join(base_dir, "..", "data", "split")

In [14]:
ruta_train = os.path.join(data_dir, "train_split.csv")
ruta_test = os.path.join(data_dir, "test_split.csv")
ruta_val = os.path.join(data_dir, "val_split.csv")

ruta_metadata = os.path.join("games_metadata.json")

In [15]:
train_set = pd.read_csv(ruta_train)
test_set = pd.read_csv(ruta_test)

train_set["hours"] = np.log1p(train_set["hours"])
test_set["hours"] = np.log1p(test_set["hours"])

regla_rating = {True: 1, False: 0}

train_set['rating'] = train_set['is_recommended'].map(regla_rating)
test_set['rating'] = test_set['is_recommended'].map(regla_rating)

In [16]:
ratings_ = test_set[test_set["rating"] == 1]
items_relevantes = test_set.groupby("user_id")["app_id"].apply(list).to_dict()


In [17]:
# Cargamos las descripciones
final_dict = {}
info_videojuegos = defaultdict(list)

set_tags = set()
with open(ruta_metadata, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        app_id = obj["app_id"]
        final_dict[app_id] = str(obj["description"])
        info_videojuegos[app_id].extend(obj["tags"])

        for tag in obj["tags"]:
            set_tags.add(tag)

In [9]:
#info_videojuegos

In [19]:
with open("info_videojuegos.json", "w", encoding="utf-8") as f:
    json.dump(info_videojuegos, f, indent=4, ensure_ascii=False)

In [20]:
descripciones = list(final_dict.values())
keys_app_id = list(final_dict.keys())

In [21]:
train_set = train_set.sort_values("user_id").reset_index(drop=True)
test_set  = test_set.sort_values("user_id").reset_index(drop=True)

In [25]:
EMBEDDING_TEXTO = "MiniLM"

In [37]:
if EMBEDDING_TEXTO == "MiniLM":
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    embeddings = model.encode(descripciones)
    embeddings_sparse = scipy.sparse.csr_matrix(embeddings)
    dict_transformados = {i: j for i, j in zip(keys_app_id, embeddings_sparse)}
    train_set["descripciones"] = train_set["app_id"].map(dict_transformados)

elif EMBEDDING_TEXTO == "Tf":
    vectorizer = TfidfVectorizer(stop_words="english")
    descripciones_train_tfid = vectorizer.fit_transform(descripciones)
    dict_transformados = {i: j for i, j in zip(keys_app_id, descripciones_train_tfid)}
    train_set["descripciones"] = train_set["app_id"].map(dict_transformados)

In [38]:
test_set["descripciones"] = test_set["app_id"].map(dict_transformados)

In [39]:
columnas_importantes = ["user_id", "app_id", "hours", "descripciones"]
train_set = train_set[columnas_importantes]
test_set = test_set[columnas_importantes]

In [40]:
test_set_user_uniques = test_set["user_id"].unique().tolist()
test_set_app_uniques = test_set["app_id"].unique().tolist()


cartesian_df = pd.DataFrame(list(product(test_set_user_uniques, test_set_app_uniques)),
                            columns=["user_id", "app_id"])

cartesian_df

,user_id,app_id
0,731,42960
1,731,1017180
2,731,352720
3,731,368260
4,731,33230
...,...,...
21297895,14304349,816090
21297896,14304349,544390
21297897,14304349,529110
21297898,14304349,39680


In [41]:
cartesian_df["descripciones"] = cartesian_df["app_id"].map(dict_transformados)

In [ ]:
columnas_train, columnas_predict = ["user_id", "app_id", "descripciones"], ["hours"]
X_train_df, y_train = train_set[columnas_train], train_set[columnas_predict]
#X_test_df, y_test = test_set[columnas_train], test_set[columnas_predict]
X_test_df = cartesian_df[columnas_train]

numerical_features_train = X_train_df[["user_id", "app_id"]].values
numerical_features_test = X_test_df[["user_id", "app_id"]].values

descripciones_sparse_train = scipy.sparse.vstack(X_train_df["descripciones"].tolist())
descripciones_sparse_test = scipy.sparse.vstack(X_test_df["descripciones"].tolist())

X_train = scipy.sparse.hstack((numerical_features_train, descripciones_sparse_train))
X_test = scipy.sparse.hstack((numerical_features_test, descripciones_sparse_test))

In [35]:
group_train = train_set.groupby("user_id").size().tolist()
group_test  = test_set.groupby("user_id").size().tolist()

In [ ]:
ranker = XGBRanker(
    objective="rank:pairwise",
    learning_rate=0.1,
    n_estimators=300,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

ranker.fit(
    X_train,
    y_train,
    group=group_train
)


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:descripciones: object

In [81]:
y_pred = ranker.predict(X_test)

In [ ]:
recomendaciones = defaultdict(list)

for i in range(X_test_df.shape[0]):
    user_id = X_test_df.iloc[i]['user_id']
    app_id = X_test_df.iloc[i]['app_id']
    predicted_score = y_pred[i]
    recomendaciones[user_id].append([app_id, predicted_score])

In [83]:
recomendaciones_def = defaultdict(list)
for usuario, lista_recomendaciones in recomendaciones.items():
    recomendaciones_ord = sorted(lista_recomendaciones, key = lambda x: x[1], reverse=True)
    recomendaciones_ord = [i[0] for i in recomendaciones_ord]
    recomendaciones_def[usuario] = recomendaciones_ord

In [114]:
precision_list = list()
recall_list = list()
ndcg_list = list()
f1_list = list()
hitrate_list = list()
map10_list = list()
diversity_list = list()

for usuario, recomendaciones_usuario in recomendaciones_def.items():
    items_rel_usuario = items_relevantes[usuario]
    recomendaciones_10 = recomendaciones_usuario[:10]
    
    precision_usuario = precision_at_k(recomendaciones_10, items_rel_usuario)
    recall_usuario = recall_at_k(recomendaciones_10, items_rel_usuario)
    ndcg_usuario = ndcg_at_k(recomendaciones_10, items_rel_usuario)
    f1_usuario = f1_at_k(recomendaciones_10, items_rel_usuario)
    hitrate_usuario = hit_score_at_k(recomendaciones_10, items_rel_usuario)
    map10_usuario = map_at_k(recomendaciones_10, items_rel_usuario)
    diversity_usuario = diversity_at_k(recomendaciones_10, info_videojuegos)
    
    precision_list.append(precision_usuario)
    recall_list.append(recall_usuario)
    ndcg_list.append(ndcg_usuario)
    f1_list.append(f1_usuario)
    hitrate_list.append(hitrate_usuario)
    map10_list.append(map10_usuario)
    diversity_list.append(diversity_usuario)

In [118]:
print(f"Precision@10: {np.mean(precision_list):.4f}")
print(f"Recall@10: {np.mean(recall_list):.4f}")
print(f"nDCG@10: {np.mean(ndcg_list):.4f}")
print(f"F1-Score@10: {np.mean(f1_list):.4f}")
print(f"Hit Score@10: {np.mean(hitrate_list):.4f}")
print(f"MAP@10: {np.mean(map10_list):.4f}")
print(f"Diversity: {np.mean(diversity_list):.4f}")

Precision@10: 0.0013
Recall@10: 0.0113
nDCG@10: 0.0065
F1-Score@10: 0.0023
Hit Score@10: 0.0132
MAP@10: 0.0049
Diversity: 77.8682
